In [2]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import sklearn
import nflreadpy as nfl

# NFL Win Prediction - Data Exploration

This notebook explores the historical NFL dataset that will eventually be used to train a binary classification model.

The initial goals are to:

1. Load the required Python libraries.
2. Load historical NFL game data.
3. Understand the structure of the dataset.
4. Identify useful columns.
5. Identify potential sources of data leakage.

In [3]:
print("pands version:", pd.__version__)
print("numpy version:", np.__version__)
print("scikit-learn version:", sklearn.__version__)

pands version: 3.0.6
numpy version: 2.5.3
scikit-learn version: 1.9.1


In [5]:
import sys 

print(sys.executable)
print(sys.version)


c:\Users\owens\anaconda3\envs\nfl-ml\python.exe
3.12.14 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:37:13) [MSC v.1942 64 bit (AMD64)]


## Load Historical NFL Schedule Data

We will begin by loading historical NFL schedules and game results.

The goal at this stage is not to build a model yet. It is to understand the structure of the dataset and determine what information is available for each game.

In [6]:
seasons = list(range(2015, 2026))

schedules = nfl.load_schedules(seasons)

In [7]:
games = schedules.to_pandas()

In [8]:
games.head()

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
0,2015_01_PIT_NE,2015,REG,1,2015-09-10,Thursday,20:30,PIT,21,NE,...,7.0,00-0022924,00-0019596,Ben Roethlisberger,Tom Brady,Mike Tomlin,Bill Belichick,Carl Cheffers,BOS00,Gillette Stadium
1,2015_01_IND_BUF,2015,REG,1,2015-09-13,Sunday,13:00,IND,14,BUF,...,15.0,00-0029668,00-0028118,Andrew Luck,Tyrod Taylor,Chuck Pagano,Rex Ryan,John Parry,BUF00,Ralph Wilson Stadium
2,2015_01_GB_CHI,2015,REG,1,2015-09-13,Sunday,13:00,GB,31,CHI,...,11.0,00-0023459,00-0024226,Aaron Rodgers,Jay Cutler,Mike McCarthy,John Fox,Craig Wrolstad,CHI98,Soldier Field
3,2015_01_KC_HOU,2015,REG,1,2015-09-13,Sunday,13:00,KC,27,HOU,...,NaN,00-0023436,00-0026625,Alex Smith,Brian Hoyer,Andy Reid,Bill O'Brien,Peter Morelli,HOU00,NRG Stadium
4,2015_01_CAR_JAX,2015,REG,1,2015-09-13,Sunday,13:00,CAR,20,JAX,...,7.0,00-0027939,00-0031407,Cam Newton,Blake Bortles,Ron Rivera,Gus Bradley,Ron Torbert,JAX00,EverBank Field


In [9]:
games.shape

(3028, 46)

In [10]:
games.columns.tolist()

['game_id',
 'season',
 'game_type',
 'week',
 'gameday',
 'weekday',
 'gametime',
 'away_team',
 'away_score',
 'home_team',
 'home_score',
 'location',
 'result',
 'total',
 'overtime',
 'old_game_id',
 'gsis',
 'nfl_detail_id',
 'pfr',
 'pff',
 'espn',
 'ftn',
 'away_rest',
 'home_rest',
 'away_moneyline',
 'home_moneyline',
 'spread_line',
 'away_spread_odds',
 'home_spread_odds',
 'total_line',
 'under_odds',
 'over_odds',
 'div_game',
 'roof',
 'surface',
 'temp',
 'wind',
 'away_qb_id',
 'home_qb_id',
 'away_qb_name',
 'home_qb_name',
 'away_coach',
 'home_coach',
 'referee',
 'stadium_id',
 'stadium']

In [11]:
games.info()

<class 'pandas.DataFrame'>
RangeIndex: 3028 entries, 0 to 3027
Data columns (total 46 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   game_id           3028 non-null   str    
 1   season            3028 non-null   int32  
 2   game_type         3028 non-null   str    
 3   week              3028 non-null   int32  
 4   gameday           3028 non-null   str    
 5   weekday           3028 non-null   str    
 6   gametime          3028 non-null   str    
 7   away_team         3028 non-null   str    
 8   away_score        3028 non-null   int32  
 9   home_team         3028 non-null   str    
 10  home_score        3028 non-null   int32  
 11  location          3028 non-null   str    
 12  result            3028 non-null   int32  
 13  total             3028 non-null   int32  
 14  overtime          3028 non-null   int32  
 15  old_game_id       3028 non-null   str    
 16  gsis              3028 non-null   int32  
 17  nfl_de

In [12]:
games["season"].value_counts().sort_index

<bound method Series.sort_index of season
2021    285
2023    285
2024    285
2025    285
2022    284
2020    269
2015    267
2016    267
2017    267
2018    267
2019    267
Name: count, dtype: int64>

In [13]:
games.sample(5)

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
1812,2021_15_KC_LAC,2021,REG,15,2021-12-16,Thursday,20:20,KC,34,LAC,...,NaN,00-0033873,00-0036355,Patrick Mahomes,Justin Herbert,Andy Reid,Brandon Staley,Shawn Smith,LAX01,SoFi Stadium
95,2015_07_PIT_KC,2015,REG,7,2015-10-25,Sunday,13:00,PIT,13,KC,...,7.0,00-0030524,00-0023436,Landry Jones,Alex Smith,Mike Tomlin,Andy Reid,Walt Coleman,KAN00,Arrowhead Stadium
2781,2025_03_LA_PHI,2025,REG,3,2025-09-21,Sunday,13:00,LA,26,PHI,...,9.0,00-0026498,00-0036389,Matthew Stafford,Jalen Hurts,Sean McVay,Nick Sirianni,Brad Rogers,PHI00,Lincoln Financial Field
32,2015_03_WAS_NYG,2015,REG,3,2015-09-24,Thursday,20:25,WAS,21,NYG,...,4.0,00-0029604,00-0022803,Kirk Cousins,Eli Manning,Jay Gruden,Tom Coughlin,Ron Torbert,NYC01,MetLife Stadium
477,2016_15_PHI_BAL,2016,REG,15,2016-12-18,Sunday,13:00,PHI,26,BAL,...,9.0,00-0032950,00-0026158,Carson Wentz,Joe Flacco,Doug Pederson,John Harbaugh,Craig Wrolstad,BAL00,M&T Bank Stadium


## Build the Game Backbone

The schedule dataset will serve as the backbone of the project.

Each row represents one NFL game. Additional team, player, matchup, and contextual features will later be attached to these rows using identifiers such as season, week, team, and game ID.

Before building features, the schedule data must be cleaned and the prediction target must be defined.

In [14]:
games["game_type"].value_counts()

game_type
REG    2895
WC       56
DIV      44
CON      22
SB       11
Name: count, dtype: int64

In [15]:
regular_games = games.loc[games["game_type"] == "REG"].copy()

In [16]:
games.loc[...]

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
0,2015_01_PIT_NE,2015,REG,1,2015-09-10,Thursday,20:30,PIT,21,NE,...,7.0,00-0022924,00-0019596,Ben Roethlisberger,Tom Brady,Mike Tomlin,Bill Belichick,Carl Cheffers,BOS00,Gillette Stadium
1,2015_01_IND_BUF,2015,REG,1,2015-09-13,Sunday,13:00,IND,14,BUF,...,15.0,00-0029668,00-0028118,Andrew Luck,Tyrod Taylor,Chuck Pagano,Rex Ryan,John Parry,BUF00,Ralph Wilson Stadium
2,2015_01_GB_CHI,2015,REG,1,2015-09-13,Sunday,13:00,GB,31,CHI,...,11.0,00-0023459,00-0024226,Aaron Rodgers,Jay Cutler,Mike McCarthy,John Fox,Craig Wrolstad,CHI98,Soldier Field
3,2015_01_KC_HOU,2015,REG,1,2015-09-13,Sunday,13:00,KC,27,HOU,...,NaN,00-0023436,00-0026625,Alex Smith,Brian Hoyer,Andy Reid,Bill O'Brien,Peter Morelli,HOU00,NRG Stadium
4,2015_01_CAR_JAX,2015,REG,1,2015-09-13,Sunday,13:00,CAR,20,JAX,...,7.0,00-0027939,00-0031407,Cam Newton,Blake Bortles,Ron Rivera,Gus Bradley,Ron Torbert,JAX00,EverBank Field
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3023,2025_20_LA_CHI,2025,DIV,20,2026-01-18,Sunday,18:30,LA,20,CHI,...,6.0,00-0026498,00-0039918,Matthew Stafford,Caleb Williams,Sean McVay,Ben Johnson,Shawn Hochuli,CHI98,Soldier Field
3024,2025_20_HOU_NE,2025,DIV,20,2026-01-18,Sunday,15:00,HOU,16,NE,...,2.0,00-0039163,00-0039851,C.J. Stroud,Drake Maye,DeMeco Ryans,Mike Vrabel,Shawn Smith,BOS00,Gillette Stadium
3025,2025_21_NE_DEN,2025,CON,21,2026-01-25,Sunday,15:00,NE,10,DEN,...,10.0,00-0039851,00-0035264,Drake Maye,Jarrett Stidham,Mike Vrabel,Sean Payton,Alex Kemp,DEN00,Empower Field at Mile High
3026,2025_21_LA_SEA,2025,CON,21,2026-01-25,Sunday,18:30,LA,27,SEA,...,2.0,00-0026498,00-0034869,Matthew Stafford,Sam Darnold,Sean McVay,Mike Macdonald,Clay Martin,SEA00,Lumen Field


In [17]:
regular_games.shape

(2895, 46)

In [18]:
backbone_columns = [
    "game_id",
    "season",
    "week",
    "gameday",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "home_rest",
    "away_rest",
    "location"
]

regular_games[backbone_columns].head(10)

,game_id,season,week,gameday,home_team,away_team,home_score,away_score,home_rest,away_rest,location
0,2015_01_PIT_NE,2015,1,2015-09-10,NE,PIT,28,21,7,7,Home
1,2015_01_IND_BUF,2015,1,2015-09-13,BUF,IND,27,14,7,7,Home
2,2015_01_GB_CHI,2015,1,2015-09-13,CHI,GB,23,31,7,7,Home
3,2015_01_KC_HOU,2015,1,2015-09-13,HOU,KC,20,27,7,7,Home
4,2015_01_CAR_JAX,2015,1,2015-09-13,JAX,CAR,9,20,7,7,Home
5,2015_01_CLE_NYJ,2015,1,2015-09-13,NYJ,CLE,31,10,7,7,Home
6,2015_01_SEA_STL,2015,1,2015-09-13,STL,SEA,34,31,7,7,Home
7,2015_01_MIA_WAS,2015,1,2015-09-13,WAS,MIA,10,17,7,7,Home
8,2015_01_NO_ARI,2015,1,2015-09-13,ARI,NO,31,19,7,7,Home
9,2015_01_DET_SD,2015,1,2015-09-13,SD,DET,33,28,7,7,Home


In [19]:
regular_games[["home_score", "away_score"]].isna().sum()

home_score    0
away_score    0
dtype: int64

In [20]:
(regular_games["home_score"] == regular_games["away_score"]).sum()

np.int64(10)

In [21]:
game_backbone = regular_games.loc[
    regular_games["home_score"] != regular_games["away_score"],
    backbone_columns
    ].copy()

In [22]:
game_backbone.head()

,game_id,season,week,gameday,home_team,away_team,home_score,away_score,home_rest,away_rest,location
0,2015_01_PIT_NE,2015,1,2015-09-10,NE,PIT,28,21,7,7,Home
1,2015_01_IND_BUF,2015,1,2015-09-13,BUF,IND,27,14,7,7,Home
2,2015_01_GB_CHI,2015,1,2015-09-13,CHI,GB,23,31,7,7,Home
3,2015_01_KC_HOU,2015,1,2015-09-13,HOU,KC,20,27,7,7,Home
4,2015_01_CAR_JAX,2015,1,2015-09-13,JAX,CAR,9,20,7,7,Home


In [23]:
game_backbone["home_win"] = (
    game_backbone["home_score"] > game_backbone["away_score"]
).astype(int)

In [25]:
game_backbone[["season", "week", "away_team", "home_team", "away_score", "home_score", "home_win"]].sample(10)

,season,week,away_team,home_team,away_score,home_score,home_win
1964,2022,5,SF,CAR,37,15,0
1614,2021,1,CLE,KC,29,33,1
2472,2024,1,LA,DET,20,26,1
2036,2022,10,ARI,LA,27,17,0
2191,2023,2,LV,BUF,10,38,1
1093,2019,2,IND,TEN,19,17,0
1497,2020,12,WAS,DAL,41,16,0
2405,2023,16,SEA,TEN,20,17,0
701,2017,12,MIA,NE,17,35,1
2657,2024,14,CAR,PHI,16,22,1


In [26]:
game_backbone["home_win"].value_counts()

home_win
1    1575
0    1310
Name: count, dtype: int64

In [27]:
game_backbone["home_win"].value_counts(normalize=True)

home_win
1    0.545927
0    0.454073
Name: proportion, dtype: float64